<a href="https://colab.research.google.com/github/JoseGG8/Estructuras-de-Datos-y-Laboratorio/blob/main/Laboratorio/Laboratorio_Hashes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Punto #1**


In [ ]:
import itertools
import hashlib

# --- CONFIGURACIÓN DEL SHA-256 ---
def calcular_sha256(texto):
    """Función auxiliar para obtener el hexadecimal de un string"""
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()

# --- ENTRADA DE DATOS (INPUT) ---
# El usuario ingresa el hash hexadecimal (ej: 5e884898da28047151d0...)
hash_objetivo = input("Introduce el hash SHA-256 objetivo: ").strip().lower()

# --- PARÁMETROS DE LA SECUENCIA ---
digitos = '0123456789'
longitud = 10

# Generador de todas las combinaciones posibles (10^10 combinaciones)
combinaciones = itertools.product(digitos, repeat=longitud)

print(f"Iniciando búsqueda de preimagen para: {hash_objetivo}...")

# --- BÚSQUEDA POR FUERZA BRUTA ---
encontrado = False

for combinacion in combinaciones:
    # Concatenamos los 10 dígitos en un solo string
    secuencia_candidata = ''.join(combinacion)

    # Calculamos el hash de la secuencia actual
    hash_candidato = calcular_sha256(secuencia_candidata)

    # Comparación de cadenas (Output si hay coincidencia)
    if hash_candidato == hash_objetivo:
        print(f"\n¡COINCIDENCIA ENCONTRADA!")
        print(f"La secuencia de 10 números es: {secuencia_candidata}")
        print(f"Hash validado: {hash_candidato}")
        encontrado = True
        break

if not encontrado:
    print("\nNo se encontró la secuencia. Verifique que el hash pertenezca a 10 dígitos.")

KeyboardInterrupt: Interrupted by user

**Punto #2**

In [ ]:
##ARBOL DE MERKLE
import hashlib

# Clase para representar cada nodo del árbol (hojas, nodos intermedios y raíz)
class Nodo:
    def __init__(self, key):
        self.key = key      # Almacena el valor hash del nodo
        self.pair = None    # Referencia al nodo hermano (complemento para el siguiente hash)

class MerkleArbol:
    def __init__(self, initial_keys):
        self.root = None
        self.initial_keys = initial_keys    # Lista de datos/transacciones originales
        self.leafs_nodos = []               # Almacenará los nodos hoja ya hasheados

    # Genera un hash SHA-256 a partir de una cadena de texto
    def hash(self, key):
        return hashlib.sha256(key.encode("utf-8")).hexdigest()

    # Concatena dos hashes y devuelve el hash del resultado (hijo izquierdo + hijo derecho)
    def hash_pairs(self, hash1, hash2):
        return self.hash(hash1 + hash2)

    # Convierte las transacciones iniciales en los nodos hoja del árbol (primer nivel de hashing)
    def hash_initialKeys(self):
        for key in self.initial_keys:
            self.leafs_nodos.append(Nodo(self.hash(key)))

    # Proceso iterativo para construir el árbol desde las hojas hasta la raíz
    def build_tree(self):
        nodes = self.leafs_nodos

        # Se repite el proceso hasta que solo quede un nodo (la raíz)
        while len(nodes) > 1:
            next_level = []

            # Si el número de nodos es impar, se duplica el último para balancear el árbol
            if len(nodes) % 2 != 0:
                nodes.append(Nodo(nodes[-1].key))

            # Agrupa los nodos de dos en dos para generar el nivel superior
            for i in range(0, len(nodes), 2):
                left = nodes[i]
                right = nodes[i+1]

                # Establece la relación de hermandad entre los nodos
                left.pair = right
                right.pair = left

                # Calcula el hash del padre a partir de los dos hijos
                parent_hash = self.hash_pairs(left.key, right.key)
                next_level.append(Nodo(parent_hash))

            # El nivel recién generado se convierte en la base para la siguiente iteración
            nodes = next_level

        # Asigna y devuelve el Merkle Root final
        self.root = nodes[0] if nodes else None
        return self.root


In [ ]:
"""
Script para la recuperación del orden de transacciones mediante Merkle Root.
"""

from itertools import permutations
import random

# --- CONFIGURACIÓN DE REFERENCIA ---
# Definimos el estado original "secreto"
TRANSACCIONES_REFERENCIA = ["hola2", "hola1", "hola4", "hola3"]
merkle_tree = MerkleArbol(TRANSACCIONES_REFERENCIA)
merkle_tree.hash_initialKeys()
root_referencia = merkle_tree.build_tree()

print(f"ROOT DE REFERENCIA: {root_referencia.key}\n")

# Mostramos las transacciones al usuario en orden aleatorio para no dar pistas
print(f"Transacciones conocidas (orden aleatorio):")
random.shuffle(TRANSACCIONES_REFERENCIA)
for i, transaccion in enumerate(TRANSACCIONES_REFERENCIA):
    print(f"{i+1}. {transaccion}")

# --- ENTRADA DE DATOS ---
TRANSACCIONES_input = input("\nIngrese las transacciones separadas por coma para iniciar búsqueda: \n")
TRANSACCIONES = [t.strip() for t in TRANSACCIONES_input.split(",")]

# --- MOTOR DE BÚSQUEDA ---
# Generamos el espacio de búsqueda (permutaciones)
permutaciones = [list(p) for p in permutations(TRANSACCIONES)]

for permutacion in permutaciones:
    # Reconstrucción tentativa del árbol
    merkle_tree_prueba = MerkleArbol(permutacion)
    merkle_tree_prueba.hash_initialKeys()
    root_a_comparar = merkle_tree_prueba.build_tree()

    # Verificación de integridad contra la referencia
    if root_referencia.key == root_a_comparar.key:
        print(f"\n[ÉXITO] Coincidencia encontrada.")
        print(f"Hash Root validado: {root_a_comparar.key}")
        break

# --- SALIDA DE RESULTADOS ---
print("\nORDEN DETERMINADO DE LAS TRANSACCIONES:")
for i, transaccion in enumerate(permutacion):
    print(f"{i+1}. {transaccion}")

el root de referencia es 210d3613b24c917d2a7ceb359ad443565eb4fbff65d1c1638235db33206b3e76 

las transacciones posibles son: 
1. hola2 
2. hola4 
3. hola1 
4. hola3 
Ingrese las transacciones separadas por coma (ej: hola1,hola2,hola3): hola1,hola2,hola3,hola4
la permutacion ['hola2', 'hola1', 'hola4', 'hola3'] de las transacciones en este orden genera el root 210d3613b24c917d2a7ceb359ad443565eb4fbff65d1c1638235db33206b3e76 

Orden de las transacciones:
1. hola2 
2. hola1 
3. hola4 
4. hola3 
